**Project Data Strategy: Synthetic Pricing Engine**

**Objective: To simulate a high-growth Electronics Retailer with 100 SKUs and 20,000 daily customer sessions to identify price optimization opportunities.**


1. Catalog Infrastructure (The 'Product Master')To create a realistic environment, I developed a product catalog based on the following constraints:

2. Inventory Depth: 100 unique SKUs across 5 core categories: Audio, Computers, Peripherals, Cameras, and Smart Home.

3. Cost Realism: Landed costs range from $30 to $800, covering both accessories and premium hardware.

4. Financial Guardrails: A strict 45% to 55% Gross Margin rule is applied to all initial pricing.

5. Constraint Formula: $Price = \frac{Cost}{1 - Margin}$

6. Business Logic: This ensures the ML model operates within "Safe" corporate profitability boundaries.

Customer Behavior Simulation (The 'Clickstream')

We generate 20,000 daily sessions using a Stochastic Funnel Model to simulate organic traffic.

Price Elasticity Rules:

1. The engine uses "Threshold Behavior" to create the signals the ML model needs to learn

2. The Sweet Spot: When the markup is below 1.9x cost, the base conversion probability is set to 3.5%.

3. Price Resistance: When markups exceed 1.9x, the probability drops to 2.5%.

4. Gaussian Noise: A noise factor ($std=0.005$) is added to every session to prevent the data from being a "perfect" math equation, forcing the model to deal with real-world variance.

The Randomized Funnel:

1. Each session is processed through a normalized weight distribution to ensure mathematical stability:

2. High-Intent Sessions: Optimized for a 3-6% final conversion (High Add-to-Cart rates).

3. Low-Intent Sessions: Optimized for a 0.1-0.4% final conversion (Window Shopping).

4. Stochastic Normalization: All funnel weights (View → Cart → Buy) are normalized per session to ensure $P(sum) = 1.0$.3.

MLOps Drift Scenario (The 'Competitor Event')

1. To validate the SageMaker Model Monitor, a secondary "Drifted" dataset was created:

2. Anomaly: The "Audio" category conversion rate is manually suppressed by 70%.

3. Scenario: This simulates a competitor launching an aggressive "Price War" or a Flash Sale on specific items.

4. Goal: The system must detect this feature-target correlation shift and trigger an automated alert in the MLOps pipeline.

Why this is a "Day Measure" this dataset represents one full business day. In the subsequent steps, we will:

1. Process these 20,000 logs to find "Friction Points".

2. Train a model to suggest price adjustments that could lift the total Conversion Rate from 3% to 4% by targeting high-interest, low-buy items.

In [ ]:
import pandas as pd
import numpy as np
import uuid

# --- SETTINGS ---
N_PRODUCTS = 100
N_SESSIONS = 25000 # Increased slightly for better statistical stability
np.random.seed(42)

# 1. CREATE PRODUCT MASTER
categories = ['Audio', 'Computers', 'Peripherals', 'Cameras', 'Smart Home']
products = []
for i in range(N_PRODUCTS):
    cat = np.random.choice(categories)
    base_cost = np.random.uniform(30, 800)
    margin = np.random.uniform(0.45, 0.55)
    price = base_cost / (1 - margin)
    products.append({
        'sku_id': f'ELEC-{1000+i}',
        'product_name': f'{cat} Device {i}',
        'category': cat,
        'base_cost': round(base_cost, 2),
        'current_msrp': round(price, 2),
        'inventory_level': np.random.randint(20, 500)
    })
df_prod = pd.DataFrame(products)

# 2. AGGRESSIVE LOG GENERATION
def generate_logs(df_p, sessions):
    logs = []
    # Force 25% of products to be "High Friction" (High Cart, Low Buy)
    friction_skus = df_p.sample(int(N_PRODUCTS * 0.25))['sku_id'].values

    for _ in range(sessions):
        prod = df_p.sample(1).iloc[0]

        if prod['sku_id'] in friction_skus:
            # High interest (12% Cart), but almost no sales (0.5% Buy)
            # This WILL trigger the target_adjustment logic
            weights = np.array([87.5, 12.0, 0.5])
        else:
            # Healthy products (Normal distribution)
            weights = np.array([75, 10, 15])

        p_dist = weights / weights.sum()

        logs.append({
            'session_id': str(uuid.uuid4())[:8],
            'sku_id': prod['sku_id'],
            'duration_sec': np.random.randint(10, 400),
            'action': np.random.choice([0, 1, 2], p=p_dist) # 0=View, 1=Cart, 2=Buy
        })
    return pd.DataFrame(logs)

# 3. GENERATE & SAVE
df_normal = generate_logs(df_prod, N_SESSIONS)
df_prod.to_csv('product_master.csv', index=False)
df_normal.to_csv('clickstream_normal.csv', index=False)

print("✅ New 'High-Friction' data generated. Upload these to S3 now.")